# 흉부 X-ray 폐렴 분류 — 열화 강건성 개선 실험

원본: [anjanatiha/Pneumonia-Detection-from-Chest-X-Ray-Images-with-Deep-Learning](https://github.com/anjanatiha/Pneumonia-Detection-from-Chest-X-Ray-Images-with-Deep-Learning)

**사용법**: 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후, 위에서부터 순서대로 실행.
각 실험이 끝날 때마다 Drive(`MyDrive/pneumonia_robustness`)에 자동 저장되므로, 런타임이 끊겨도 0·1·2번만 다시 실행하고 **3번(복구)**을 돌리면 이어서 할 수 있음.

## 0. 환경 + Drive

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/pneumonia_robustness
%cd /content
!rm -rf repo && git clone -q https://github.com/20241112-svg/Pneumonia-Detection-from-Chest-X-Ray-Images-with-Deep-Learning.git repo
%cd repo/improved
!pip install -q -r requirements.txt

## 1. Kaggle 데이터
`kaggle.json`(Kaggle → Settings → API → **Create Legacy API Key**) 을 업로드.

In [ ]:
from google.colab import files
import os
os.makedirs('/root/.kaggle', exist_ok=True)
files.upload()
!mv kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p data/raw -q
!unzip -q -o data/raw/chest-xray-pneumonia.zip -d data/raw

## 2. 분할 (test 624 고정, train 10% → val)

In [ ]:
!rm -rf data/input && python make_split.py --src data/raw/chest_xray --dst data/input --val-ratio 0.1 --seed 42

## 3. (런타임이 끊겼을 때만) Drive 에서 이전 결과 복구
처음 실행이면 건너뜀.

In [ ]:
!cp -r /content/drive/MyDrive/pneumonia_robustness/runs /content/drive/MyDrive/pneumonia_robustness/results . 2>/dev/null; ls runs results 2>/dev/null || echo '복구할 결과 없음(처음 실행)'

## 3-1. (v2 코드로 다시 돌리기 전 한 번만) 1차 결과 보관
1차(11/16~17) 결과는 `results_v1/` 과 Drive `v1/` 에 남기고, 같은 tag 로 다시 학습.

In [ ]:
!mkdir -p results_v1 /content/drive/MyDrive/pneumonia_robustness/v1
!cp results/*.csv results/*.json results/*.png results_v1/ 2>/dev/null; cp -r runs results /content/drive/MyDrive/pneumonia_robustness/v1/ 2>/dev/null
!ls results_v1 | head

## 4. 열화 샘플 확인 (그림만)

In [ ]:
import glob
sample = glob.glob('data/input/test/NORMAL/*')[0]
!python degrade.py --image "{sample}" --out degrade_grid.png
from IPython.display import Image; Image('degrade_grid.png', width=900)

## 5. 베이스라인 (원본 + BatchNorm) — 약 10분
원본 CNN 그대로는 TF2에서 학습이 안 됨(loss 0.693 고정, 전부 폐렴 예측 → acc 0.625). 그 실패 재현은 맨 아래 9번.

**v2 (11/18) 변경**: 같은 seed → 같은 결과가 되도록 샘플별 증강 시드 + TF determinism 적용. 학습 후 깨끗한 train 으로 BN 통계 재보정(`best.keras`), 재보정 전 모델은 `best_norecal.keras`.

In [ ]:
!python train_baseline.py --data data/input --tag base --seed 0 --bn
!python eval_robustness.py --model runs/base_s0/best.keras --data data/input --tag base_s0
!cp -r runs results /content/drive/MyDrive/pneumonia_robustness/ && echo saved
from IPython.display import Image; Image('results/base_s0.png', width=700)

## 6. 개선 실험 — 각 10~12분
이미 Drive 에 결과가 있는 실험은 건너뛰어도 됨 (`ls results`).

**v2 변경**: 열화 증강이 강도 1~4, 2종 합성(확률 0.5)으로 바뀜. 1차에서 블러 고강도·홀드아웃 열화에 개선이 없던 원인 대응.

In [ ]:
# 6-1. CLAHE 전처리만
!python train_baseline.py --data data/input --tag clahe --seed 0 --bn --clahe
!python eval_robustness.py --model runs/clahe_s0/best.keras --data data/input --tag clahe_s0
!cp -r runs results /content/drive/MyDrive/pneumonia_robustness/ && echo saved

In [ ]:
# 6-2. 열화 증강만 (JPEG·모션블러는 홀드아웃이라 학습에 안 씀)
!python train_baseline.py --data data/input --tag aug --seed 0 --bn --degrade-aug
!python eval_robustness.py --model runs/aug_s0/best.keras --data data/input --tag aug_s0
!cp -r runs results /content/drive/MyDrive/pneumonia_robustness/ && echo saved

In [ ]:
# 6-3. 둘 다
!python train_baseline.py --data data/input --tag both --seed 0 --bn --clahe --degrade-aug
!python eval_robustness.py --model runs/both_s0/best.keras --data data/input --tag both_s0
!cp -r runs results /content/drive/MyDrive/pneumonia_robustness/ && echo saved

## 6-4. 구조 변경 — 사전학습 ViT-S/16 + LoRA (각 10~15분)
원본 CNN 을 ImageNet 사전학습 ViT 로 바꾸되 LoRA 어댑터(약 60만 파라미터)만 학습. 학습이 끝나면 열화 평가까지 자동으로 수행.

In [ ]:
!python train_lora_vit.py --data data/input --tag vit_lora --seed 0 --mode lora
!cp -r runs results /content/drive/MyDrive/pneumonia_robustness/ && echo saved

In [ ]:
!python train_lora_vit.py --data data/input --tag vit_lora_both --seed 0 --mode lora --clahe --degrade-aug
!cp -r runs results /content/drive/MyDrive/pneumonia_robustness/ && echo saved

In [ ]:
# 비교군 (시간 되면): head 만 학습 / 전체 미세조정
# !python train_lora_vit.py --data data/input --tag vit_head --seed 0 --mode head
# !python train_lora_vit.py --data data/input --tag vit_full --seed 0 --mode full
# !cp -r runs results /content/drive/MyDrive/pneumonia_robustness/ && echo saved

In [ ]:
!python eval_robustness.py --compare results/base_s0.csv results/both_s0.csv results/vit_lora_s0.csv results/vit_lora_both_s0.csv --out results/compare_vit.png
!cp -r runs results /content/drive/MyDrive/pneumonia_robustness/ && echo saved
!cat results/vit_*_summary.json
from IPython.display import Image; Image('results/compare_vit.png', width=1000)

## 7. 비교

In [ ]:
!python eval_robustness.py --compare results/base_s0.csv results/clahe_s0.csv results/aug_s0.csv results/both_s0.csv --out results/compare.png
!cp -r runs results /content/drive/MyDrive/pneumonia_robustness/ && echo saved
!cat results/*_summary.json
from IPython.display import Image; Image('results/compare.png', width=1000)

## 8. 분석 — 임계값 복구 + 열화 강도 근거 (GPU 불필요, 2~3분)
`analyze_threshold.py`: 열화별로 acc@0.5 / acc@val-임계값 / acc@oracle / AUC 를 비교 → "붕괴가 정보 손실인지 보정 붕괴인지".
`data_stats.py`: 데이터셋 자체의 노이즈·대비·선명도 분포 위에 각 열화 강도가 어디 놓이는지.

In [ ]:
!python analyze_threshold.py --probs results/base_s0_probs.npz results/aug_s0_probs.npz results/both_s0_probs.npz --tag cnn
from IPython.display import Image; display(Image('results/threshold_cnn.png', width=1100))

In [ ]:
!python data_stats.py --data data/input --out results/data_stats
from IPython.display import Image; display(Image('results/data_stats.png', width=1000))
!cp -r runs results /content/drive/MyDrive/pneumonia_robustness/ && echo saved

## 9. 시드 반복 — base / both / vit_lora × seed 1, 2 (약 1시간)
seed 0 은 위에서 이미 돌렸으므로 1, 2 만. 최종 보고는 3 seed 평균 ± 표준편차.

In [ ]:
for s in [1, 2]:
    !python train_baseline.py --data data/input --tag base --seed {s} --bn
    !python eval_robustness.py --model runs/base_s{s}/best.keras --data data/input --tag base_s{s}
    !python train_baseline.py --data data/input --tag both --seed {s} --bn --clahe --degrade-aug
    !python eval_robustness.py --model runs/both_s{s}/best.keras --data data/input --tag both_s{s}
    !cp -r runs results /content/drive/MyDrive/pneumonia_robustness/ && echo saved
# ViT 도 시드 반복 (각 10~15분)
# for s in [1, 2]:
#     !python train_lora_vit.py --data data/input --tag vit_lora_both --seed {s} --mode lora --clahe --degrade-aug
#     !cp -r runs results /content/drive/MyDrive/pneumonia_robustness/ && echo saved

## 10. (보고서용) 원본 CNN 그대로 — 학습 실패 재현
약 10분. 결과: loss 0.693 고정, test acc 0.625, AUC 0.5.

In [ ]:
# !python train_baseline.py --data data/input --tag base_orig --seed 0
# !cat runs/base_orig_s0/log.csv
# !cp -r runs results /content/drive/MyDrive/pneumonia_robustness/ && echo saved